# Notebook 03 — Camada Silver (Transformação e Qualidade)

## Objetivos da Camada Silver

A camada Silver aplica transformações de **limpeza, padronização e enriquecimento** sobre os dados brutos da Bronze:

1. **Limpeza**: Remover registros com chaves nulas e duplicatas
2. **Padronização**: Normalizar capitalização, remover espaços extras, padronizar formatos
3. **Tipagem**: Converter colunas para tipos corretos (datas, decimais, inteiros)
4. **Enriquecimento**: Calcular campos derivados (ex: `total_pedido`)

Os dados resultantes são salvos em tabelas Delta no diretório `silver/`.


## 1. Criação da SparkSession


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, upper, lower, when, count, countDistinct, round as spark_round
from pyspark.sql.types import DecimalType, IntegerType, DateType

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
BRONZE_DELTA_DIR = os.path.join(DATA_DIR, "bronze_delta")
SILVER_DIR = os.path.join(DATA_DIR, "silver")

spark = (
    SparkSession.builder
    .appName("NB03_Silver")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession iniciada. Versão: {spark.version}")


## 2. Silver — Clientes

Transformações aplicadas:
- `nome` e `cidade`: trim + title case
- `estado`: uppercase
- `email`: lowercase
- `data_cadastro`: cast para date
- Remover duplicatas por `cliente_id`
- Remover registros com `cliente_id` nulo


In [ ]:
df_bronze_clientes = spark.read.format("delta").load(
    os.path.join(BRONZE_DELTA_DIR, "bronze_clientes")
)

print("=== ANTES (Bronze) ===")
df_bronze_clientes.show(5, truncate=False)

df_silver_clientes = df_bronze_clientes \
    .withColumn("nome", initcap(trim(col("nome")))) \
    .withColumn("cidade", initcap(trim(col("cidade")))) \
    .withColumn("estado", upper(trim(col("estado")))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("data_cadastro", col("data_cadastro").cast(DateType())) \
    .filter(col("cliente_id").isNotNull()) \
    .dropDuplicates(["cliente_id"])

print("\n=== DEPOIS (Silver) ===")
df_silver_clientes.show(5, truncate=False)

silver_clientes_path = os.path.join(SILVER_DIR, "clientes")
df_silver_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_clientes_path)

print(f"\n[OK] silver/clientes salvo com {df_silver_clientes.count()} registros")


## 3. Silver — Produtos

Transformações aplicadas:
- `sku`: trim + uppercase
- `nome_produto`, `categoria`, `subcategoria`: trim + title case
- `preco_venda` e `preco_custo`: cast para decimal(10,2)
- Remover registros com SKU nulo ou preços inválidos (<= 0)
- Remover duplicatas por SKU


In [ ]:
df_bronze_produtos = spark.read.format("delta").load(
    os.path.join(BRONZE_DELTA_DIR, "bronze_produtos")
)

print("=== ANTES (Bronze) ===")
df_bronze_produtos.show(5, truncate=False)

df_silver_produtos = df_bronze_produtos \
    .withColumn("sku", upper(trim(col("sku")))) \
    .withColumn("nome_produto", initcap(trim(col("nome_produto")))) \
    .withColumn("categoria", initcap(trim(col("categoria")))) \
    .withColumn("subcategoria", initcap(trim(col("subcategoria")))) \
    .withColumn("preco_venda", col("preco_venda").cast(DecimalType(10, 2))) \
    .withColumn("preco_custo", col("preco_custo").cast(DecimalType(10, 2))) \
    .filter(col("sku").isNotNull()) \
    .filter(col("preco_venda") > 0) \
    .filter(col("preco_custo") > 0) \
    .dropDuplicates(["sku"])

print("\n=== DEPOIS (Silver) ===")
df_silver_produtos.show(5, truncate=False)

silver_produtos_path = os.path.join(SILVER_DIR, "produtos")
df_silver_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_produtos_path)

print(f"\n[OK] silver/produtos salvo com {df_silver_produtos.count()} registros")


## 4. Silver — Pedidos (com Enriquecimento)

Transformações aplicadas:
- `data_pedido`: cast para date
- `quantidade`: cast para integer
- `valor_frete`: cast para decimal(10,2)
- Remover registros com chaves nulas (`pedido_id`, `cliente_id`, `sku`)
- Remover duplicatas por `pedido_id`
- **Enriquecimento**: JOIN com `silver/produtos` para obter `preco_venda`
- **Cálculo**: `total_pedido = quantidade * preco_venda`


In [ ]:
df_bronze_pedidos = spark.read.format("delta").load(
    os.path.join(BRONZE_DELTA_DIR, "bronze_pedidos")
)

print("=== ANTES (Bronze) ===")
df_bronze_pedidos.show(5, truncate=False)

df_silver_produtos_read = spark.read.format("delta").load(
    os.path.join(SILVER_DIR, "produtos")
).select("sku", "preco_venda")

df_silver_pedidos = df_bronze_pedidos \
    .withColumn("data_pedido", col("data_pedido").cast(DateType())) \
    .withColumn("quantidade", col("quantidade").cast(IntegerType())) \
    .withColumn("valor_frete", col("valor_frete").cast(DecimalType(10, 2))) \
    .filter(col("pedido_id").isNotNull()) \
    .filter(col("cliente_id").isNotNull()) \
    .filter(col("sku").isNotNull()) \
    .dropDuplicates(["pedido_id"]) \
    .join(df_silver_produtos_read, on="sku", how="left") \
    .withColumn("total_pedido", col("quantidade") * col("preco_venda"))

print("\n=== DEPOIS (Silver com total_pedido) ===")
df_silver_pedidos.select(
    "pedido_id", "cliente_id", "sku", "data_pedido",
    "quantidade", "preco_venda", "total_pedido", "valor_frete"
).show(10, truncate=False)

silver_pedidos_path = os.path.join(SILVER_DIR, "pedidos")
df_silver_pedidos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_pedidos_path)

print(f"\n[OK] silver/pedidos salvo com {df_silver_pedidos.count()} registros")


## 5. Encerramento da SparkSession


In [ ]:
spark.stop()
print("SparkSession encerrada.")


## Resumo da Camada Silver

| Tabela | Transformações Aplicadas | Registros |
|--------|--------------------------|-----------|
| **clientes** | trim, capitalização, dedup, filtro de nulos | 1.500 |
| **produtos** | trim, capitalização, cast decimal, filtro de valores inválidos | 250 |
| **pedidos** | cast de tipos, dedup, JOIN com produtos, cálculo `total_pedido` | 8.000 |

Os dados estão agora limpos, padronizados e enriquecidos. No próximo notebook (**NB04**), faremos a **Modelagem Dimensional** na Camada Gold (Star Schema).
